In [20]:
import os
import pandas as pd
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from tqdm import tqdm
from typing import List, Literal, List, Dict, Any, Optional
import numpy as np
import seaborn as sns

from datasets import load_dataset
import random
import json
import re
from functools import partial
from datasets import Dataset
from copy import deepcopy
import evaluate
import nltk
from scipy.stats import ttest_ind
import string
from collections import Counter

import openai
import os
import time
import pandas as pd

from ragas.llms import LangchainLLMWrapper
from langchain_deepseek import ChatDeepSeek
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import AnswerAccuracy
from dotenv import load_dotenv
load_dotenv()

True

In [21]:
frames_benchmark = load_dataset("google/frames-benchmark")['test']

In [22]:
frames_benchmark[0]

{'Unnamed: 0': 0,
 'Prompt': "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ",
 'Answer': 'Jane Ballou',
 'wikipedia_link_1': 'https://en.wikipedia.org/wiki/President_of_the_United_States',
 'wikipedia_link_2': 'https://en.wikipedia.org/wiki/James_Buchanan',
 'wikipedia_link_3': 'https://en.wikipedia.org/wiki/Harriet_Lane',
 'wikipedia_link_4': 'https://en.wikipedia.org/wiki/List_of_presidents_of_the_United_States_who_died_in_office',
 'wikipedia_link_5': 'https://en.wikipedia.org/wiki/James_A._Garfield',
 'wikipedia_link_6': None,
 'wikipedia_link_7': None,
 'wikipedia_link_8': None,
 'wikipedia_link_9': None,
 'wikipedia_link_10': None,
 'wikipedia_link_11+': None,
 'reasoning_types': 'Multiple constraints',
 'wiki_links': "['https://en.wikipedia.org/wiki/President_of_the_United_States', 'https://en.wikipedia.org/wiki/

In [23]:
def simplify(example):
    return {
        "id": example["Unnamed: 0"],
        "request": example["Prompt"],
        "answer": example["Answer"]
    }

simplified_frames_benchmark = frames_benchmark.map(simplify, remove_columns=frames_benchmark.column_names)
print(simplified_frames_benchmark[0])

Map:   0%|          | 0/824 [00:00<?, ? examples/s]

{'id': 0, 'request': "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ", 'answer': 'Jane Ballou'}


In [28]:
simplified_frames_benchmark.to_json("frames_sample_raw.jsonl",orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

183450

## Using off-the-shelf Qwen3-4B for classification

In [ ]:
frames_sample = load_dataset(
    "json",
    data_files="frames_sample_raw.jsonl",
    split="train"  # 必须指定 split，否则默认返回 DatasetDict
)

In [25]:
from helper_functions_qa import (prepare_test_prompts, run_experiment, merge_df_into_dataset_by_order)

In [26]:
system_prompt = """
You are an expert analyst. Your task is to analyze and determine whether an input user query is "fully specified" or "underspecified".

"""

task_FS_UND = """
Analyze the following input user query:

{"query": "TARGET"}

Please provide your analysis in the following JSON format:

{"query": "TARGET", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}
"""

In [ ]:
Qwen3_4B = "Qwen/Qwen3-4B"

# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(Qwen3_4B, padding_side='left')
model = AutoModelForCausalLM.from_pretrained(Qwen3_4B)

# 将模型移到可用设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
model = model.to(device)

In [29]:
df = pd.read_json("frames_sample_raw.jsonl", lines=True)
test_prompts = prepare_test_prompts(df, task_FS_UND)
print(test_prompts[0])

Start preparing prompts...
# Testing data points: 824
Generation complete: 824 prompts
Average prompt length: 531 bytes (~132 tokens)

Analyze the following input user query:

{"query": "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? "}

Please provide your analysis in the following JSON format:

{"query": "If my future wife has the same first name as the 15th first lady of the United States' mother and her surname is the same as the second assassinated president's mother's maiden name, what is my future wife's name? ", "reasoning": "[YOUR_DETAILED_REASONING]", "judgment": "[fully specified/underspecified]"}



In [ ]:
test_df = run_experiment(test_prompts, df_GoogleNQ_sample)
test_df.to_csv('./intermediate/BASELINE_frames_classified.csv')
test_df

In [ ]:
annotated_dataset = merge_df_into_dataset_by_order(
    frames_sample, 
    test_df, 
    columns=["thinking", "model_response", "model_pred"],
    prefix="qwen3_",
    inplace=False
)

In [ ]:
# Create the UND subset
underspecified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "underspecified"
)

# Create the FS subset
fully_specified_set = annotated_dataset.filter(
    lambda x: x["qwen3_model_pred"].strip().lower() == "fully specified"
)

# The size of subsets
print(f"Underspecified samples: {len(underspecified_set)}")
print(f"Fully specified samples: {len(fully_specified_set)}")

In [ ]:
annotated_dataset.to_json("./intermediate/BASELINE_classified_frames_sample_all.jsonl", orient="records", lines=True)
underspecified_set.to_json("./intermediate/BASELINE_classified_frames_sample_UND.jsonl", orient="records", lines=True)
fully_specified_set.to_json("./intermediate/BASELINE_classified_frames_sample_FS.jsonl", orient="records", lines=True)